# 회귀 실습

**Regression**

연속적인 수치 값을 예측하는 지도학습 문제.

소재 분야에서 이해하기: 소재의 밴드갭을 eV 단위로 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 연속값 예측

경도(HV)처럼 연속적인 값을 예측하는 문제입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
for name, model in [('선형회귀', LinearRegression()),
                    ('랜덤 포레스트', RandomForestRegressor(n_estimators=300, random_state=0))]:
    pred = model.fit(X_train, y_train).predict(X_test)
    print('%-12s MAE %.2f HV / R2 %.3f' % (name, mean_absolute_error(y_test, pred), r2_score(y_test, pred)))

In [ ]:
model = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)
residual = y_test - model.predict(X_test)
plt.scatter(model.predict(X_test), residual, s=16)
plt.axhline(0, color='k', lw=1)
plt.xlabel('predicted hardness (HV)'); plt.ylabel('residual (HV)')
plt.title('residual plot'); plt.show()
print('잔차 평균 %.2f, 표준편차 %.2f' % (residual.mean(), residual.std()))

## 2. 해석

잔차가 0 주변에 고르게 퍼져 있으면 모델이 남긴 편향이 크지 않다는 뜻입니다. 특정 구간에서
잔차가 한쪽으로 치우치면 그 구간의 관계를 모델이 못 잡고 있다는 신호입니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#regression)을 여세요.